In [ ]:
import os
import time
import json
import pandas as pd
import schedule
from datetime import datetime
from dotenv import load_dotenv
from scraper import scrape_getquin
from ai_engine import analyze_portfolio

# Controllo se siamo in un ambiente Jupyter per una visualizzazione migliore
try:
    from IPython.display import display
    IN_JUPYTER = True
except ImportError:
    IN_JUPYTER = False

# Carica le variabili d'ambiente dal file .env
# Se non esiste, prova a caricare da .env.example
if os.path.exists(".env"):
    load_dotenv(".env")
elif os.path.exists(".env.example"):
    load_dotenv(".env.example")
else:
    load_dotenv()

def job():
    print(f"\n--- Inizio monitoraggio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ---")
    
    # Leggi i dati iniziali del portafoglio dal file .env (se presenti)
    initial_value = os.getenv("INITIAL_PORTFOLIO_VALUE", "55000")
    start_date = os.getenv("PORTFOLIO_START_DATE", "2025-09-01")
    
    # 1. Estrazione dati
    print("Ricerca screenshot del portafoglio...")
    scrape_result = scrape_getquin()
    
    if scrape_result["status"] == "error":
        print(f"Errore:\n{scrape_result['message']}")
        return
        
    image_path = scrape_result.get("image_path")
    
    # 2. Analisi AI
    print("Analisi del portafoglio con Gemini in corso (potrebbe richiedere un minuto per analizzare tutti i titoli)...")
    try:
        analysis_json = analyze_portfolio(image_path, initial_value=initial_value, start_date=start_date)
        
        # Pulizia stringa JSON se Gemini ha aggiunto markdown
        if analysis_json.startswith("```json"):
            analysis_json = analysis_json[7:]
        if analysis_json.endswith("```"):
            analysis_json = analysis_json[:-3]
            
        data = json.loads(analysis_json.strip())
    except Exception as e:
        print(f"Errore durante l'analisi AI o il parsing JSON:\n{str(e)}")
        # Stampa la risposta grezza per debug
        try:
            print(f"Risposta grezza: {analysis_json}")
        except:
            pass
        return
        
    # 3. Creazione DataFrame e Output
    print("\n" + "="*80)
    print("📊 RIASSUNTO PORTAFOGLIO")
    print("="*80)
    
    summary = data.get("portfolio_summary", {})
    df_summary = pd.DataFrame([summary])
    
    if "strategy_summary" in df_summary.columns:
        strategy = df_summary["strategy_summary"].iloc[0]
        df_summary_display = df_summary.drop(columns=["strategy_summary"])
    else:
        strategy = "N/A"
        df_summary_display = df_summary.copy()
        
    # Formattazione percentuale per il summary
    if "percentage_return" in df_summary_display.columns:
        df_summary_display["percentage_return"] = df_summary_display["percentage_return"].apply(lambda x: f"{x}%" if pd.notna(x) and str(x).strip() != "" else x)
        
    if IN_JUPYTER:
        display(df_summary_display)
    else:
        print(df_summary_display.to_markdown(index=False))
    
    print(f"\n💡 Strategia e Conclusioni:\n{strategy}\n")
    
    print("="*100)
    print("📈 ANALISI ASSET E SENTIMENT")
    print("="*100)
    
    assets = data.get("assets", [])
    df_assets = pd.DataFrame(assets)
    
    if not df_assets.empty:
        df_assets_display = df_assets.copy()
        
        # Formattazione percentuale per gli asset
        if "weight_percentage" in df_assets_display.columns:
            df_assets_display["weight_percentage"] = df_assets_display["weight_percentage"].apply(lambda x: f"{x}%" if pd.notna(x) and str(x).strip() != "" else x)
        if "profit_loss_percent" in df_assets_display.columns:
            df_assets_display["profit_loss_percent"] = df_assets_display["profit_loss_percent"].apply(lambda x: f"{x}%" if pd.notna(x) and str(x).strip() != "" else x)
            
        # Mostriamo le colonne principali per la tabella
        display_cols = ["name", "ticker", "position_value", "profit_loss_eur", "profit_loss_percent", "weight_percentage", "sentiment"]
        available_cols = [c for c in display_cols if c in df_assets_display.columns]
        
        if IN_JUPYTER:
            display(df_assets_display[available_cols])
        else:
            print(df_assets_display[available_cols].to_markdown(index=False))
        
        print("\n📰 Dettaglio News Sentiment:")
        news_cols = ["name", "sentiment", "news_summary"]
        available_news_cols = [c for c in news_cols if c in df_assets_display.columns]
        
        if IN_JUPYTER:
            display(df_assets_display[available_news_cols])
        else:
            print(df_assets_display[available_news_cols].to_markdown(index=False))
    else:
        print("Nessun asset trovato.")
    
    print("="*100 + "\n")
    
    # 4. Salva i DataFrame in CSV nella cartella output
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    summary_file = os.path.join(output_dir, f"portfolio_summary_{timestamp}.csv")
    assets_file = os.path.join(output_dir, f"portfolio_assets_{timestamp}.csv")
    
    try:
        df_summary.to_csv(summary_file, index=False, sep=";")
        df_assets.to_csv(assets_file, index=False, sep=";")
        print(f"✅ Dati salvati con successo in formato CSV in:")
        print(f"   - {summary_file}")
        print(f"   - {assets_file}")
    except Exception as e:
        print(f"Errore durante il salvataggio dei file CSV: {e}")
        
    print("\nProcesso completato con successo.")

In [16]:
job()


--- Inizio monitoraggio: 2026-03-18 13:06:40 ---
Ricerca screenshot del portafoglio...
Cerco lo screenshot più recente del portafoglio nella cartella 'input'...
Trovato screenshot più recente: input\portafoglio_18032026_114207.png
Analisi del portafoglio con Gemini in corso (potrebbe richiedere un minuto per analizzare tutti i titoli)...
Caricamento immagine e richiesta a Gemini in corso...

📊 RIASSUNTO PORTAFOGLIO


,initial_value,current_total_value,absolute_return,percentage_return
0,55000.0,59482.16,4482.16,8.15%



💡 Strategia e Conclusioni:
Il portafoglio ha generato un rendimento positivo significativo dall'inizio dell'anno, superando il valore iniziale. La diversificazione tra materie prime (oro, petrolio) ed equity (tecnologia, Europa, Italia) ha contribuito a questa performance. In particolare, l'oro ha mostrato una forte crescita. Le posizioni in giganti tecnologici come Microsoft e Amazon, nonostante un sentiment di mercato a volte misto, mantengono prospettive positive grazie all'innovazione AI e alle solide valutazioni degli analisti. Alcuni titoli europei e italiani, come Nexi, Intesa Sanpaolo e Ferrari, stanno affrontando venti contrari o un sentiment più cauto, suggerendo la necessità di un monitoraggio continuo e di potenziali aggiustamenti strategici per ottimizzare la performance e mitigare i rischi in queste aree.

📈 ANALISI ASSET E SENTIMENT


""
0
1
2
3
4
5
6
7
8
9



📰 Dettaglio News Sentiment:


,name,sentiment,news_summary
0,Invesco Physical Gold ETC,Bullish,L'ETC ha registrato afflussi significativi in ...
1,Xtrackers Stoxx Europe 600 ETI,N/A,N/A
2,WisdomTree Brent Crude Oil 2x,Bullish,I prezzi del petrolio Brent hanno superato i 1...
3,Invesco Physical Silver ETC,N/A,N/A
4,Xtrackers S&P 500 ETF,Mixed to Bullish,Il sentiment di mercato per le azioni statunit...
5,Siemens Energy AG,Bullish,Siemens Energy AG ha un consenso di rating 'Bu...
6,Unipol Gruppo,N/A,N/A
7,Microsoft,Mixed to Bullish,Le azioni Microsoft hanno mostrato un moderato...
8,Amazon,Mixed to Moderate Buy,Amazon mostra un sentiment misto sulle opzioni...
9,Intesa Sanpaolo,Mixed to Moderate Buy,Intesa Sanpaolo ha un consenso 'Moderate Buy' ...



✅ Dati salvati con successo in formato CSV in:
   - output\portfolio_summary_20260318_130732.csv
   - output\portfolio_assets_20260318_130732.csv

Processo completato con successo.
